In [ ]:
import sys
sys.path.append('/Users/antonk/py/sentinel1denoised')
from s1denoise import Sentinel1Image

import os
import glob

from warping import *
from nansat import Nansat, Domain, NSR

from cartopy import crs as ccrs
import xarray as xr

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.util import view_as_windows

In [ ]:
x_res, y_res = 500, 500
ps = ccrs.NorthPolarStereo(central_longitude=-45, true_scale_latitude=70)
asip_dir = '/Users/antonk/Data/dmi_asip/SEAICE_ARC_PHY_AUTO_L3_MYNRT_011_023/cmems_obs-si_arc_phy_my_l3_P1D_202411'

In [ ]:
idir = '/Users/antonk/Data/s1/'
ifiles = sorted(glob.glob(f'{idir}/*SAFE'))

for s1file in ifiles:
    n1 = Nansat(s1file)
    n1.reproject_gcps()
    n1.vrt.tps = True

    n1c_lon, n1c_lat = n1.get_corners()
    n1cx, n1cy = ps.transform_points(ccrs.PlateCarree(), n1c_lon, n1c_lat)[:, :2].T

    xmin, xmax = float(n1cx.min() - x_res), float(n1cx.max() + x_res)
    ymin, ymax = float(n1cy.min() - y_res), float(n1cy.max() + y_res)

    asip_mask = n1.time_coverage_start.strftime('%Y/%m/*_%Y%m%d.nc')
    asip_file = glob.glob(f'{asip_dir}/{asip_mask}')[0]
    with xr.open_dataset(asip_file) as ds:
        xc_ascending = ds.xc.values[0] < ds.xc.values[-1]
        yc_ascending = ds.yc.values[0] < ds.yc.values[-1]

        xc_slice = slice(xmin, xmax) if xc_ascending else slice(xmax, xmin)
        yc_slice = slice(ymin, ymax) if yc_ascending else slice(ymax, ymin)

        sic_subset = ds["sic"].sel(xc=xc_slice, yc=yc_slice)[0].values

    src_dom = Domain(
        '+proj=stere +lat_0=90 +lat_ts=70 +lon_0=-45 +x_0=0 +y_0=0 +a=6378273 +b=6356889.449 +units=m +no_defs',
        f'-te {(xmin - x_res/2):.2f} {(ymin - y_res/2):.2f} {(xmax + x_res/2):.2f} {(ymax + y_res/2):.2f} -tr {x_res} {y_res}'
        )
    n1.resize(0.25)
    dst_img = warp(src_dom, sic_subset, n1)


    i = Sentinel1Image(s1file)
    s0 = {}
    for pol in ['HH', 'HV']:
        s0[pol] = i.remove_thermal_noise(pol)
    s0db = {}
    for pol in ['HH', 'HV']:
        s0db[pol] = 10*np.log10(s0[pol])
    s0dbf = {}
    for pol in ['HH', 'HV']:
        s0dbf[pol] = gaussian_filter(s0db[pol], 2, truncate=2)[::4, ::4]

    sub_size = sub_step = np.array(dst_img.shape) // 2
    windows = {}
    for pol in ['HH', 'HV']:
        windows[pol] = view_as_windows(s0dbf[pol], sub_size, sub_step)
    windows['sic'] = view_as_windows(dst_img, sub_size, sub_step)

    p_lim = 0.1
    for i in range(windows['HH'].shape[0]):
        for j in range(windows['HH'].shape[1]):
            r = windows['HV'][i, j].copy()
            r_min, r_max = np.nanpercentile(r[r > -40], [p_lim, 100-p_lim])
            r = np.clip((r - r_min) / (r_max - r_min), 0, 1)
            print(r_min, r_max)

            b = windows['HH'][i, j].copy()
            b_min, b_max = np.nanpercentile(b, [p_lim, 100-p_lim])
            b = np.clip((b - b_min) / (b_max - b_min), 0, 1)
            print(b_min, b_max)

            #g = r * (r + 2 * b * (1 - r))
            g = np.clip(windows['sic'][i, j] / 100, 0, 1)

            img = np.dstack([np.where(np.isnan(x), 0, 1+254*x).astype(np.uint8) for x in (r, g, b)])
            plt.imsave(f'collocated/{os.path.basename(s1file)}#{i}_{j}.png', img)
